In [1]:
from datasets import load_dataset

# Загрузка всего датасета
dataset = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k")

# Посмотрим, какие сплиты доступны
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'clean', 'noisy'],
        num_rows: 11572
    })
    test: Dataset({
        features: ['id', 'clean', 'noisy'],
        num_rows: 824
    })
})


In [2]:
import torch

def segment_audio(audio, segment_len=16000):
    segments = []
    for start in range(0, len(audio), segment_len):
        seg = audio[start:start+segment_len]
        if len(seg) < segment_len:
            # дополняем нулями до нужной длины
            seg = torch.cat([seg, torch.zeros(segment_len - len(seg))])
        segments.append(seg)
    return segments

In [3]:
from torch.utils.data import Dataset

class VoiceBankDataset(Dataset):
    def __init__(self, hf_dataset, split="train", segment_len=16000):
        self.dataset = hf_dataset[split]
        self.segment_len = segment_len
        self.examples = []

        # создаём сегменты заранее
        for idx in range(len(self.dataset)):
            example = self.dataset[idx]
            noisy = torch.tensor(example["noisy"]["array"], dtype=torch.float32)
            clean = torch.tensor(example["clean"]["array"], dtype=torch.float32)
            noisy_segs = segment_audio(noisy, segment_len)
            clean_segs = segment_audio(clean, segment_len)
            self.examples.extend(list(zip(noisy_segs, clean_segs)))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [4]:
from torch.utils.data import DataLoader

train_dataset = VoiceBankDataset(dataset, split="train", segment_len=16000)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, drop_last=True)

In [5]:
import torch.nn as nn

class SimpleDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(16, 1, kernel_size=15, padding=7)
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, seq_len)
        x = self.net(x)
        return x.squeeze(1)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleDenoiser().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(3):  # пример 3 эпох
    epoch_loss = 0
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()
        output = model(noisy)
        loss = criterion(output, clean)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}, Avg Loss: {epoch_loss/len(train_loader):.6f}")

Epoch 1, Avg Loss: 0.000740
Epoch 2, Avg Loss: 0.000681
Epoch 3, Avg Loss: 0.000669


In [7]:
torch.save(model.state_dict(), "simple_denoiser.pth")

In [9]:
import torch
import soundfile as sf
from pathlib import Path

# --- Параметры ---
device = "cuda" if torch.cuda.is_available() else "cpu"
input_file = "input_audio.wav"      # файл, который ты положишь в папку
output_file = "denoised_output.wav" # имя файла для результата

# --- Загружаем модель ---
model = SimpleDenoiser().to(device)
model.load_state_dict(torch.load("simple_denoiser.pth", map_location=device))
model.eval()

# --- Загружаем аудио ---
audio_array, sr = sf.read(input_file)
audio_tensor = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0).to(device)  # batch=1

# --- Прогон через модель ---
with torch.no_grad():
    denoised = model(audio_tensor).cpu().squeeze(0).numpy()

# --- Сохраняем результат ---
sf.write(output_file, denoised, sr)
print(f"Очищенный файл сохранён как {output_file}")

Очищенный файл сохранён как denoised_output.wav
